In [18]:
%pip install -r ../requirements.txt requests 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Helper function - shows no output when run. It only defines the function
import os                # check whether a file already exists
import requests          # download data from the internet
import pandas as pd      # work with tables (DataFrames)

# Police/CBS table with monthly registered crimes per neighbourhood (47022NED)
URL = "https://dataderden.cbs.nl/ODataFeed/OData/47022NED/TypedDataSet"

def download(filter_text):
    """Download all rows that match the filter. The API returns max 10,000 rows per request,
    so we download page by page until there are no more rows."""
    rows, skip = [], 0   # rows: collects all rows from all pages | skip: rows already downloaded
    while True:          # keep going until we stop the loop with "break"
        # Request parameters: which rows, JSON format, max 10,000 rows per page, skip the rows we already have
        params = {"$filter": filter_text, "$format": "json", "$top": 10000, "$skip": skip}
        # Send the request; give up if the server does not respond within 120 seconds
        response = requests.get(URL, params=params, timeout=120)
        # Stop with a clear error message if the server returns an error (e.g. 500 Server Error)
        response.raise_for_status()
        # Convert the JSON response to Python; the rows are stored under the key "value"
        page = response.json()["value"]
        # An empty page means there are no more rows -> stop the loop
        if not page:
            break
        rows += page     # add the rows from this page to the collected list
        skip += 10000    # move 10,000 rows ahead to the next page
        print(f"{len(rows):,} rows fetched")   # show progress
    # Convert the list of rows to a table (DataFrame) and return it
    return pd.DataFrame(rows)

In [2]:
# Check that the URL points to ODataFeed (not ODataApi)
print(URL)

https://dataderden.cbs.nl/ODataFeed/OData/47022NED/TypedDataSet


In [3]:
# '1.1.1 ' = residential burglary (the trailing space is required)
# 'BU0363' = all neighbourhoods in Amsterdam
test = download("SoortMisdrijf eq '1.1.1 ' and substringof('BU0363', WijkenEnBuurten)")
print(test.shape)   # (number of rows, number of columns)
test.head()         # first 5 rows

10,000 rows fetched
20,000 rows fetched
30,000 rows fetched
40,000 rows fetched
50,000 rows fetched
60,000 rows fetched
70,000 rows fetched
80,000 rows fetched
90,000 rows fetched
91,509 rows fetched
(91509, 5)


,ID,SoortMisdrijf,WijkenEnBuurten,Perioden,GeregistreerdeMisdrijven_1
0,3481590,1.1.1,BU0363AA01,2012MM01,0
1,3481591,1.1.1,BU0363AA01,2012MM02,0
2,3481592,1.1.1,BU0363AA01,2012MM03,0
3,3481593,1.1.1,BU0363AA01,2012MM04,1
4,3481594,1.1.1,BU0363AA01,2012MM05,0


### Test the function on one municipality first
Downloading the whole country takes several minutes, so we first test the function on one
municipality. We use Amsterdam because the result is easy to check (about 500 neighbourhoods).
Note: the later decision to focus the project on Amsterdam is a separate choice (see notebook 01).

In [4]:
# Output file for the full dataset (.gz = compressed, keeps the file below GitHub's 100 MB limit)
OUTPUT_FILE = "../data/raw/47022NED_burglary_monthly_buurt.csv.gz"

if os.path.exists(OUTPUT_FILE):
    # File already exists -> load it from disk instead of downloading it again
    burglary=pd.read_csv(OUTPUT_FILE)
else:
    # File does not exist: download residential burglary ('1.1.1 ') for all neighbourhoods in the country ('BU')
    burglary = download("SoortMisdrijf eq '1.1.1 ' and substringof('BU', WijkenEnBuurten)")
    burglary.to_csv(OUTPUT_FILE, index=False)   # index=false: do not save the pandas row numbers as an extra column
    print("Downloaded and saved to",OUTPUT_FILE)

print(burglary.shape)   # (number of rows, number of columns)
burglary.head()         # first 5 rows



(2607210, 5)


,ID,SoortMisdrijf,WijkenEnBuurten,Perioden,GeregistreerdeMisdrijven_1
0,3274677,1.1.1,BU16800000,2012MM01,1
1,3274678,1.1.1,BU16800000,2012MM02,3
2,3274679,1.1.1,BU16800000,2012MM03,0
3,3274680,1.1.1,BU16800000,2012MM04,1
4,3274681,1.1.1,BU16800000,2012MM05,0


In [5]:
n_buurten = burglary["WijkenEnBuurten"].nunique() # number of distinct neighbourhoods
n_months = burglary["Perioden"].nunique()          # number of distinct months

print("Neighbourhoods:", n_buurten)
print("Months:", n_months, "| from", burglary["Perioden"].min(), "to", burglary["Perioden"].max())
print("Neighbourhoods x months =", n_buurten * n_months, "| rows in table =", len(burglary)) # should be equal if there are no missing rows
print("Missing values per column")
print(burglary.isna().sum())   # number of empty/missing cells per column
print("Total registered residential burglaries:", int(burglary["GeregistreerdeMisdrijven_1"].sum()))




Neighbourhoods: 14730
Months: 177 | from 2012MM01 to 2026MM09
Neighbourhoods x months = 2607210 | rows in table = 2607210
Missing values per column
ID                            0
SoortMisdrijf                 0
WijkenEnBuurten               0
Perioden                      0
GeregistreerdeMisdrijven_1    0
dtype: int64
Total registered residential burglaries: 661753
